# Start

In [3]:
# Taruh ini di sel paling atas sendiri di Notebook-mu
%load_ext autoreload
%autoreload 2

In [4]:
%%writefile ./src/__init__.py
# 

Overwriting ./src/__init__.py


## sensor_simulation.py

In [5]:
%%writefile ./src/sensor_simulation.py
from datetime import datetime, timezone, timedelta
import random

def sensor_simulation(dummy_size_kb: int = 1) -> dict[str,float]:
  tz_wib = timezone(timedelta(hours=7))
  temperature = random.uniform(20.0, 40.0)
  air_humidity = random.uniform(40.0, 100.0)
  soil_moisture = random.uniform(0.0, 100.0)
  soil_ph = random.uniform(4.0, 7.0)

  return {
      'reading_timestamp': datetime.now(tz_wib).isoformat(),
      'temperature': temperature,
      'air_humidity': air_humidity,
      'soil_moisture': soil_moisture,
      'soil_ph': soil_ph,
      'dummy' : 'x' * 1024 * dummy_size_kb

      ## if prefer lower decimal count to save space
      # round('temperature': temperature, 3),
      # round('air_humidity': air_humidity, 3),
      # round('soil_moisture': soil_moisture, 3),
      # round('soil_ph': soil_ph, 3)
  }

Overwriting ./src/sensor_simulation.py


In [6]:
from src.sensor_simulation import sensor_simulation

print(sensor_simulation())

{'reading_timestamp': '2026-06-09T13:43:45.112612+07:00', 'temperature': 22.60259766846338, 'air_humidity': 50.13473855266591, 'soil_moisture': 96.53524004241515, 'soil_ph': 6.598284711163664, 'dummy': 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx

In [7]:
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

print(AESGCM.generate_key(bit_length=256))

b'T&2\x01\x9b\xa8\x8d|6x\xbb\x18\x11\x8d\n8\xc5=\xae$j\xfe\xf1\x0c\x13\xd9\xe0T^:\x8d\xb4'


## rsa_encryption.py

In [8]:
%%writefile ./src/rsa_encryption.py
from cryptography.hazmat.primitives.asymmetric import padding as asym_padding
from cryptography.hazmat.primitives.serialization import load_pem_public_key
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import time


def prepare_rsa_key(public_key_pem: str) -> tuple[bytes, bytes, float]:
    start = time.perf_counter()
    session_key = AESGCM.generate_key(bit_length=256)
    end = time.perf_counter()
    
    public_key = load_pem_public_key(public_key_pem.encode("utf-8"))

    aad_component = public_key.encrypt(
        session_key,
        asym_padding.OAEP(
            mgf=asym_padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )
    return session_key, aad_component, (end-start)

Overwriting ./src/rsa_encryption.py


In [9]:
from src.rsa_encryption import prepare_rsa_key
from src.server_public_key import server_rsa_public_key

print(prepare_rsa_key(server_rsa_public_key))

(b'\x0bH\x05;\x1fQ\x18I\xe2\xd3\xea\x9a\xe7\xdb\xce]\\s8\xf6\xa2 \xdfK\xe2;\xdb\xdf\xfa\x8c\xf9\xbc', b'*p6t\xfb\x9f2\x93\xbb\xa4\xbf\xc5\x8e>x\x93\xad4X\x98\x08\xda\xf0\xb8\xbc\xba\xc1\xa3\xd4\x04\xab\xa2\x1c\xb8\xe8\xd3\xd7?Q\x81\xbe\xc7\r\x9c\xc1\x02\xaa\xf5O\x05\xea\xa0\xcb\x91\x99\x86\xfa\xd7\xcdK\xb5\x14W\xd1\xf1\xe0E\xa5T\xeaQnP4\x17f\x10\x89\xf5P-\x04T\x0b(V\x82G\xa1,_\x86M+QD= \x8a\xd7\xef\xc4:\x91\x1e{\xde\xc7\x9b\x04\xfc\\\xfd\xabUb\xe6\x06\xda\xff\xe75\x07\xc2\x95\xf2A\xeb\x99\x86G\x13\x19VL\xb5\xb0\xde\x9f\xcf\xb8k\x8dj\xef\xf7\xef{\x16\x9c\x13\x19\xa6&\t\x99\xa9\xb6;z\x84\xd5\x0c=k\xb9\xe1\r\xa0e\xe4d2`\xde\x0e \x1f\xa7\xd0(SS\xc8\xcb-`\xc3\x96\xe4\xa1Ik[*\x98\x04\x14\x0e\xc9\x8a\xd6\xf6\x88I\xf0e\x8f\xdd\xd3\xfd\xcc\xf5a{\x8b&\xe7\xe2\xae/\xc6\xd5cl\xc7Z\xa1\xa5\'\x0e\x88Q\xce\x11C\xd8x\xfc\xdf[\xfa+\x1d\xb2\x8d"\x8f\xfb\x19\xe8PL\xd4\x8a~', 2.180004958063364e-05)


## ecc_encryption.py

In [10]:
%%writefile ./src/ecc_encryption.py
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey
from cryptography.hazmat.primitives.serialization import load_pem_public_key, Encoding, PublicFormat
from cryptography.hazmat.primitives.kdf.hkdf import HKDF
from cryptography.hazmat.primitives.hashes import SHA256
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import time

def prepare_ecc_key(public_key_pem: str) -> tuple[bytes, bytes, float]:
    server_public = load_pem_public_key(public_key_pem.encode("utf-8"))

    start = time.perf_counter()
    ephemeral_private = X25519PrivateKey.generate()
    

    ephemeral_public  = ephemeral_private.public_key()
    ephemeral_public_bytes = ephemeral_public.public_bytes(Encoding.DER, PublicFormat.SubjectPublicKeyInfo)

    shared_secret = ephemeral_private.exchange(server_public)
    session_key = HKDF(
        algorithm=SHA256(),
        length=32,
        salt=None,
        info=b"sensor-server-v1"
    ).derive(shared_secret)
    end = time.perf_counter()
    return session_key, ephemeral_public_bytes, (end-start)

Overwriting ./src/ecc_encryption.py


## build_payload.py

In [ ]:
%%writefile ./src/build_payload.py
from datetime import datetime, timezone, timedelta
import os
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
import base64
import json
from sensor_simulation import sensor_simulation
import sqlite3
import requests
from cryptography.hazmat.primitives.asymmetric.x25519 import X25519PrivateKey, X25519PublicKey

class EdgeGateway:
	def __init__(self, sensor_id, return_aad_bytes: bool = False):
		self.sensor_id = sensor_id
		self.return_aad_bytes = return_aad_bytes
		self.db_conn = sqlite3.connect(f"sensor{sensor_id}.db")
		cursor_init = self.db_conn.cursor()
		# cursor_init.execute("""
		# 	CREATE TABLE IF NOT EXISTS self_id (
		# 	id INTEGER PRIMARY KEY)
		# """)
		cursor_init.execute("""
			CREATE TABLE IF NOT EXISTS session_keys (
			id			INTEGER	PRIMARY KEY,
			key 		TEXT	NOT NULL,
			times_used	INTEGER	NOT NULL)
		""")
		cursor_init.execute("""
			CREATE TABLE IF NOT EXISTS sensor_readings (
			id			INTEGER	PRIMARY KEY,
			data		TEXT	NOT NULL,
			sent_status	INTEGER	NOT NULL)
		""")
		# cursor_init.execute("INSERT OR IGNORE INTO self_id (id) VALUES (?)", (sensor_id,))
		self.db_conn.commit()

	# def get_latest_session_key(self):
	# 	cursor = self.db_conn.cursor()
	# 	cursor.execute("""
	# 			SELECT id, key FROM session_keys ORDER BY id DESC LIMIT 1
	# 		""")
	# 	return cursor.fetchone()

	def get_latest_session_key(self):
		return self.db_conn.execute("""
				SELECT id, key FROM session_keys ORDER BY id DESC LIMIT 1
			""").fetchone()

	

	def build_payload(self, mode: str, session_key, key) -> dict:
		tz_wib = timezone(timedelta(hours=7))

		# this one gonna cascade depending on the result. None is not a valid session_key. You need to use the current session_key
		# session_key = get_session_key() # -> raw_bytes
		# session_key for testing because get_session_key isn't finished yet
		
		aad = {
			'sensor_id': self.sensor_id,
			'mode': mode,
			'transmission_timestamp': datetime.now(tz_wib).isoformat(),
			'key': base64.b64encode(key).decode('utf-8') 
		}
		aad_bytes = json.dumps(aad, separators=(',', ':'), sort_keys=True).encode('utf-8')

		aesgcm = AESGCM(session_key)
		iv = os.urandom(12)

		plaintext_string = self.db_conn.execute("""
			SELECT data FROM sensor_readings
			WHERE sent_status = 0
			ORDER BY id ASC
			LIMIT 1
		""").fetchone()[0]
		# print(plaintext_string)

		plaintext_bytes = plaintext_string.encode('utf-8') 
		# print(plaintext_bytes)
		encrypted_raw = aesgcm.encrypt(iv, plaintext_bytes, aad_bytes)
		ciphertext = encrypted_raw[:-16]
		tag = encrypted_raw[-16:]

		return {
			'aad': aad if not self.return_aad_bytes else aad_bytes.decode('utf-8'),
			'nonce': base64.b64encode(iv).decode('utf-8'),
			'ciphertext': base64.b64encode(ciphertext).decode('utf-8'),
			'tag': base64.b64encode(tag).decode('utf-8')
		}

	def post(self, url: str, *, json: dict):
		response=requests.post(url, json=json)
		if response.status_code == 200:
			self.db_conn.execute("""
				UPDATE sensor_readings
				SET sent_status = 1
				WHERE id = (
					SELECT id
					FROM sensor_readings
					WHERE sent_status = 0
					ORDER BY id ASC
					LIMIT 1
				)
			""")
			self.db_conn.commit()
		return response

	def generate_sensor_data(self, dummy_size_kb: int = 1):
		plaintext_string = json.dumps(sensor_simulation(dummy_size_kb))# -> dict[str,float]

		self.db_conn.execute("""
			INSERT INTO sensor_readings (data, sent_status)
			VALUES (?, 0)
		""", (plaintext_string,))
		self.db_conn.commit()



Overwriting ./src/build_payload.py


## server_public_key.py

In [12]:
%%writefile ./src/server_public_key.py
import os

certs_dir = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'certs'))

with open(os.path.join(certs_dir, 'rsa_public_key.pem'), 'r', encoding='utf-8') as f:
    server_rsa_public_key = f.read()

with open(os.path.join(certs_dir, 'ecc_public_key.pem'), 'r', encoding='utf-8') as f:
    server_ecc_public_key = f.read()

# BASE_DIR 
# print({"BASE_DIR": BASE_DIR})

# BASE_DIR2 = os.path.abspath(__file__)
# print({"BASE_DIR2": BASE_DIR2})


Overwriting ./src/server_public_key.py


In [13]:
from src.server_public_key import server_rsa_public_key, server_ecc_public_key
print(server_rsa_public_key)
print(server_ecc_public_key)

-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAw6L4H3IpxaTQb/JS5x3j
aJ4eFhLMToc4wfK4VH5v5e9hJHWbdh28o+RxfEHS4+/Ls9+mxP/gY7JP/6C4Z52w
S60/ZDePQqesUjjs3kfe4oQ35ql9RuHUosPKZeg/zTIQD7W5IdCGonDC+R1NbFI+
xjNv/5vLB+cTIl4U0QxOxXsNI7OyLliedZGySrr+6btfio54ZHg921Q0NpaRQawU
3SxuoeQZrtoWl7K2g7x2beO5jtcKb92u9+F6+dWBqo2YQ9NgD+owOS4XFgZ0LG7G
t7R4a5NRKHFku1vA7P7Sk/B9qHWO87weA7CFpRgkh7quBmhIvc06p7sCUQEqn//X
iwIDAQAB
-----END PUBLIC KEY-----

-----BEGIN PUBLIC KEY-----
MCowBQYDK2VuAyEAr+QCbmou7w3VCxy4mldVaX90G/blgwlackBeXqrtHmQ=
-----END PUBLIC KEY-----



## Send payload

In [15]:
%%writefile main.py
import requests
import time
# from build_payload2 import build_payload2
# from get_public_key import get_public_key
import numpy as np
from scipy import stats
import json
import pandas as pd

from src.rsa_encryption import prepare_rsa_key
from src.ecc_encryption import prepare_ecc_key
from src.build_payload import EdgeGateway
from src.server_public_key import server_rsa_public_key, server_ecc_public_key

url = 'http://localhost:3000/telemetry'

sensor1 = EdgeGateway(1)
modes = ['rsa', 'ecc']
sizes = [1, 10, 100]

for mode in modes:
    for size in sizes:
        durations_encryption = []
        packet_size = []
        generate_durations = []
        decryption_durations = []

        for _ in range(30):
            sensor1.generate_sensor_data()

        start_tp = time.perf_counter()
        for _ in range(30):
            start_encrypt = time.perf_counter()
            session_key, key, generate_dur = prepare_rsa_key(server_rsa_public_key) if mode == 'rsa' else prepare_ecc_key(server_ecc_public_key)
            payload = sensor1.build_payload(mode, session_key, key)
            end_encrypt = time.perf_counter()
            
            decryption_duration=sensor1.post(url, json=payload).json()['decryption_duration']
            
            durations_encryption.append(end_encrypt - start_encrypt)
            decryption_durations.append(decryption_duration)
            packet_size.append(len(json.dumps(payload).encode()))
            generate_durations.append(generate_dur)
        end_tp = time.perf_counter()

        df = pd.DataFrame({'Encryption (second)': durations_encryption,
                           'decryption (second)': decryption_durations,
                           'size (byte)': packet_size,
                           'session key generation (second)': generate_durations})
        df.to_csv(f'{mode} {size}kb.csv',index=False)

        df_tp = pd.DataFrame({'time (second)': [start_tp], 'time after 30 messages sent (second)': [end_tp]})
        df_tp.to_csv(f'{mode} {size}kb throughput.csv',index=False)

        print(f'{mode} {size}kb')

        d1 = np.array(durations_encryption) * 1000
        print(f"Encrypt- mean: {np.mean(d1):.3f} ms, median: {np.median(d1):.3f} ms, std: {np.std(d1):.3f} ms")

        d2 = np.array(decryption_durations) * 1000
        print(f"Decrypt- mean: {np.mean(d2):.3f} ms, median: {np.median(d2):.3f} ms, std: {np.std(d2):.3f} ms")

        print(f"Throughput: {30/(end_tp-start_tp)} message/second")

        d3 = np.array(packet_size)
        print(f"JSON size - mean: {np.mean(d3):.3f} bytes, median: {np.median(d3):.3f} bytes, std: {np.std(d3):.3f} bytes")

        d4 = np.array(generate_durations) * 1000
        print(f"Generate key- mean: {np.mean(d4):.3f} ms, median: {np.median(d4):.3f} ms, std: {np.std(d4):.3f} ms")

        print('='*80+'\n')

Writing main.py


# Stop

In [119]:
sensor1.db_conn.execute(
    """DROP TABLE IF EXISTS sensor_readings"""
)
sensor1.db_conn.commit()